In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import re

from scipy import sparse
from scipy.stats import zscore, spearmanr, pearsonr, mannwhitneyu, kruskal
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests

In [ ]:
#GD2 NR VS R
os.chdir('/home/ST_Data/MICS/GD2/260210_export_ROI1_NR_ROI2_R/spatial_h5ad_results')

In [98]:
#GD2 CART vs Inhibitor
os.chdir('/home/ST_Data/MICS/GD2/CART_VS_inhibitor/spatial_h5ad_results')

In [121]:
#KA CART vs Inhibitor
os.chdir('/home/ST_Data/MICS/KA_0305/spatial_h5ad_results')

In [134]:
adata = sc.read_h5ad(
    'all_ROI_spatial_processed_merged.h5ad'
)

In [131]:
adata.obs[['sample']].value_counts()

sample                  
KA_CART_ROI2_2              39095
KA_CART_ROI2_5              37660
KA_CART_ROI2_7              37652
KA_inhibitor_0307_ROI2_4    33274
KA_CART_ROI2_6              32937
KA_inhibitor_0307_ROI2_3    29045
KA_CART_ROI2_4              18878
KA_inhibitor_0307_ROI2_5    17132
KA_CART_ROI2_1              16118
KA_inhibitor_0307_ROI2_2    14092
KA_CART_ROI2_3_c4           11228
KA_CART_ROI2_3_c3           10557
Name: count, dtype: int64

In [135]:
# 如果 cell_id 在 adata.obs.index 里
adata.obs["cell_id"] = adata.obs.index.astype(str)

#adata.obs["ROI_group"] = adata.obs["cell_id"].str.extract(r"^(ROI\d+_\d+)", expand=False)

'''
adata.obs[["ROI_group", "group"]] = adata.obs["cell_id"].str.extract(
    r"^(ROI\d+_\d+)_(cart|inhibitor)_",
    expand=True
)
'''

adata.obs[["ROI_group", "group"]] = adata.obs["cell_id"].str.extract(
    r"^KA_(CART|inhibitor_0307)_(ROI\d+_\d+)_",
    expand=True
)


adata.obs["ROI_group_simple"] = (
    adata.obs["ROI_group"]
    .str.replace("ROI_", "", regex=False)
)

print(adata.obs[["cell_id", "ROI_group", "ROI_group_simple"]].head())
print(adata.obs["ROI_group_simple"].value_counts())

                                         cell_id ROI_group ROI_group_simple
cell_id                                                                    
KA_CART_ROI2_1_cell_1@1  KA_CART_ROI2_1_cell_1@1      CART             CART
KA_CART_ROI2_1_cell_2@1  KA_CART_ROI2_1_cell_2@1      CART             CART
KA_CART_ROI2_1_cell_3@1  KA_CART_ROI2_1_cell_3@1      CART             CART
KA_CART_ROI2_1_cell_4@1  KA_CART_ROI2_1_cell_4@1      CART             CART
KA_CART_ROI2_1_cell_5@1  KA_CART_ROI2_1_cell_5@1      CART             CART
ROI_group_simple
CART              204125
inhibitor_0307     93543
Name: count, dtype: int64


In [103]:
adata.obs[["ROI", "condition"]] = adata.obs["sample"].str.rsplit("_", n=1, expand=True)

In [129]:
print(adata.obs["ROI_group"].value_counts())

ROI_group
CART    204125
Name: count, dtype: int64


In [136]:
# =========================
# 0. 参数设置
# =========================

h5ad_file = "all_ROI_spatial_processed_merged.h5ad"

outdir = "spatial_reanalysis_density"
plot_dir = os.path.join(outdir, "spatial_plots")
result_dir = os.path.join(outdir, "results")
heatmap_dir = os.path.join(outdir, "heatmaps")

os.makedirs(outdir, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)
os.makedirs(result_dir, exist_ok=True)
os.makedirs(heatmap_dir, exist_ok=True)

sample_col = "sample"
#sample_col='ROI'
# 如果 obs 里有 group / condition / treatment，可以改这里
# 例如 group_col = "group"
#group_col = "sample"
group_col = "ROI_group"
marker_keyword = "Cell Intensity Average"
exclude_keywords = ["DAPI"]

arcsinh_cofactor = 5
leiden_resolution = 0.5
n_neighbors_leiden = 15

# 共定位分析中，每个细胞寻找多少个空间邻居
spatial_k = 30


def safe_name(x):
    x = str(x)
    x = re.sub(r"[^\w\-.]+", "_", x)
    return x.strip("_")


def short_marker_name(x):
    return (
        str(x)
        .replace(" Cell Intensity Average", "")
        .replace("Cell Intensity Average", "")
        .replace(" Cell Intensity Average", "")
        .replace("Cell Intensity Average", "")
        .strip()
    )


# =========================
# 1. 读取数据
# =========================

#adata = sc.read_h5ad(h5ad_file)

print(adata)
print("obs columns:")
print(adata.obs.columns.tolist())


# =========================
# 2. 检查 sample 和 group
# =========================

if sample_col not in adata.obs.columns:
    raise ValueError(f"obs 中没有找到 sample 列：{sample_col}")

if group_col not in adata.obs.columns:
    print(f"Warning: obs 中没有找到 {group_col}，将使用 sample 作为 group。")
    group_col = sample_col

adata.obs[sample_col] = adata.obs[sample_col].astype(str)
adata.obs[group_col] = adata.obs[group_col].astype(str)


# =========================
# 3. 检查 spatial 坐标
# =========================
# Scanpy 的 spatial embedding 通常存在 adata.obsm["spatial"]
# 如果没有，会尝试从 obs 中自动寻找 x/y 坐标列。

if "spatial" not in adata.obsm_keys():
    possible_x = ["x", "X", "center_x", "centroid_x", "spatial_x", "image_x"]
    possible_y = ["y", "Y", "center_y", "centroid_y", "spatial_y", "image_y"]

    x_col = next((c for c in possible_x if c in adata.obs.columns), None)
    y_col = next((c for c in possible_y if c in adata.obs.columns), None)

    if x_col is None or y_col is None:
        raise ValueError(
            "没有找到 adata.obsm['spatial']，也没有在 obs 中自动识别到 x/y 坐标列。"
            "请手动指定 spatial 坐标。"
        )

    adata.obsm["spatial"] = adata.obs[[x_col, y_col]].values
    print(f"Using obs columns as spatial coordinates: {x_col}, {y_col}")


# =========================
# 4. 自动提取 marker
# =========================

marker_cols = [
    col for col in adata.obs.columns
    if marker_keyword in col
    and not any(ex.lower() in col.lower() for ex in exclude_keywords)
]

if len(marker_cols) == 0:
    raise ValueError(
        f"没有找到包含 '{marker_keyword}' 的 marker 列。"
        "请检查 obs 中 marker 列名。"
    )

print(f"Found {len(marker_cols)} markers:")
print(marker_cols)


# =========================
# 5. arcsinh + sample 内 z-score
# =========================

for marker in marker_cols:
    raw_col = f"{marker}_raw"
    arcsinh_col = f"{marker}_arcsinh"
    zscore_col = f"{marker}_zscore"

    adata.obs[raw_col] = pd.to_numeric(adata.obs[marker], errors="coerce")

    adata.obs[arcsinh_col] = np.arcsinh(
        adata.obs[raw_col] / arcsinh_cofactor
    )

    # sample 内 z-score
    adata.obs[zscore_col] = (
        adata.obs
        .groupby(sample_col, observed=False)[arcsinh_col]
        .transform(
            lambda x: (x - x.mean()) / x.std(ddof=0)
            if x.std(ddof=0) > 0
            else 0
        )
    )

zscore_cols = [f"{m}_zscore" for m in marker_cols]
marker_short_names = [short_marker_name(m) for m in marker_cols]

# z-score 矩阵保存到 obsm
X_z = adata.obs[zscore_cols].astype(float).fillna(0).values
adata.obsm["X_marker_zscore"] = X_z

print("Z-score matrix shape:", adata.obsm["X_marker_zscore"].shape)

AnnData object with n_obs × n_vars = 297668 × 17
    obs: 'Cell Id', 'Cell 3D Volume', 'Cell Bbox Bottom', 'Cell Bbox Left', 'Cell Bbox Right', 'Cell Bbox Top', 'Cell Bbox X Size', 'Cell Bbox Y Size', 'Cell Center X', 'Cell Center Y', 'Cell Center Y Inverted', 'Cell Contour Bending Energy', 'Cell Contour Length', 'Cell Convex Hull Area', 'Cell Convex Hull Perimeter', 'Cell Convexity', 'Cell Cytoplasm Area', 'Cell Feret Diameter Max', 'Cell Feret Diameter Maximum Angle', 'Cell Feret Diameter Min', 'Cell Feret Diameter Minimum Angle', 'Cell Feret Diameter Perpendicular Min', 'Cell P2A', 'Cell Radius Maximum', 'Cell Radius Mean', 'Cell Radius Minimum', 'Cell Radius Standard Deviation', 'Cell Roundness', 'Cell Shape Circle Like', 'Cell Shape Ellipse Like', 'Cell Shape Elongation', 'Cell Shape Square Like', 'Cell Shape Triangle Like', 'Cell Size', 'Nuc X', 'Nuc Y', 'Nuc Y Inv', 'Nucleus 3D Volume', 'Nucleus Box Bottom', 'Nucleus Box Height', 'Nucleus Box Left', 'Nucleus Box Right', 'Nucleus

In [107]:
adata.obs[['ROI']]=adata.obs[['sample']]

In [137]:
# =========================
# 5b. CAR-T density analysis: CAR-T cells/mm2
# =========================

from scipy.spatial import cKDTree
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

density_dir = os.path.join(outdir, "cart_density_analysis")
os.makedirs(density_dir, exist_ok=True)

# ---------
# 需要你根据真实 marker 名称确认
# ---------
mcherry_marker = next((m for m in marker_cols if "cherry".lower() in m.lower()), None)
gfp_marker = next((m for m in marker_cols if "GFP".lower() in m.lower()), None)
cdh5_marker = next((m for m in marker_cols if "CDH5(VE-adhesion)".lower() in m.lower()), None)

if mcherry_marker is None:
    raise ValueError("没有找到 mCherry marker，请检查 marker_cols 中的列名。")

if gfp_marker is None:
    print("Warning: 没有找到 GFP marker，将只用 mCherry 定义 CAR-T。")

if cdh5_marker is None:
    print("Warning: 没有找到 CDH5 marker，无法做 vascular/extravascular 分析。")

print("Using markers:")
print("mCherry:", mcherry_marker)
print("GFP:", gfp_marker)
print("CDH5:", cdh5_marker)

# ---------
# 使用 arcsinh 后的 raw intensity 定义阳性细胞
# 不建议用 sample 内 z-score 作为最终阈值，因为 density/count 需要跨 sample 可比较
# ---------
mcherry_arcsinh = f"{mcherry_marker}_arcsinh"
gfp_arcsinh = f"{gfp_marker}_arcsinh" if gfp_marker is not None else None
cdh5_arcsinh = f"{cdh5_marker}_arcsinh" if cdh5_marker is not None else None

# ---------
# 阈值策略：
# 方案 A：如果有 negative/control sample，建议用 negative controls 定义阈值
# 方案 B：没有 negative control 时，使用全数据固定 percentile threshold
# 注意：所有 sample 使用同一个阈值，避免每张图自适应导致组间不可比
# ---------
mcherry_threshold = adata.obs[mcherry_arcsinh].quantile(0.99)

if gfp_arcsinh is not None:
    gfp_threshold = adata.obs[gfp_arcsinh].quantile(0.90)
else:
    gfp_threshold = None

if cdh5_arcsinh is not None:
    cdh5_threshold = adata.obs[cdh5_arcsinh].quantile(0.90)
else:
    cdh5_threshold = None

print("Thresholds:")
print("mCherry threshold:", mcherry_threshold)
print("GFP threshold:", gfp_threshold)
print("CDH5 threshold:", cdh5_threshold)

# ---------
# 定义 segmented cells
# CAR-T: mCherry+
# 如果 GFP 信号可能串色，建议用 mCherry+ and GFP-low
# ---------
adata.obs["CAR_T_mCherry_pos"] = adata.obs[mcherry_arcsinh] > mcherry_threshold

if gfp_arcsinh is not None:
    adata.obs["Tumor_GFP_pos"] = adata.obs[gfp_arcsinh] > gfp_threshold
    
    # 更保守的 CAR-T 定义：mCherry high 且 GFP 不高
    adata.obs["CAR_T"] = (
        adata.obs["CAR_T_mCherry_pos"] &
        ~adata.obs["Tumor_GFP_pos"]
    )
else:
    adata.obs["Tumor_GFP_pos"] = False
    adata.obs["CAR_T"] = adata.obs["CAR_T_mCherry_pos"]

if cdh5_arcsinh is not None:
    adata.obs["Vascular_CDH5_pos"] = adata.obs[cdh5_arcsinh] > cdh5_threshold
else:
    adata.obs["Vascular_CDH5_pos"] = False

Using markers:
mCherry: cherry Cell Intensity Average
GFP: GFP Cell Intensity Average
CDH5: CDH5(VE-adhesion) Cell Intensity Average
Thresholds:
mCherry threshold: 2.366995585581927
GFP threshold: 5.509300614532905
CDH5 threshold: 1.9031638457856093


In [138]:
# =========================
# 5b-2. Calculate analyzed segmented tissue area using Cell Size
# =========================

area_col = "Cell Size"

if area_col not in adata.obs.columns:
    raise ValueError(f"没有找到 {area_col}，请检查 obs columns。")

adata.obs[area_col] = pd.to_numeric(adata.obs[area_col], errors="coerce")

print("Cell Size summary:")
print(adata.obs[area_col].describe())

# =========================
# 请确认 Cell Size 单位
# =========================
# 如果 Cell Size 已经是 um2，用 "um2"
# 如果 Cell Size 是 pixel2，用 "pixel2"，并设置真实 pixel_size_um

cell_size_unit = "um2"

# 如果是 pixel2，改成：
# cell_size_unit = "pixel2"
# pixel_size_um = 0.325  # replace with real MACSima pixel size

if cell_size_unit == "um2":
    adata.obs["cell_area_um2_for_density"] = adata.obs[area_col]

elif cell_size_unit == "pixel2":
    pixel_size_um = 0.325  # replace with real MACSima pixel size
    adata.obs["cell_area_um2_for_density"] = (
        adata.obs[area_col] * pixel_size_um ** 2
    )

else:
    raise ValueError("cell_size_unit must be 'um2' or 'pixel2'")

adata.obs.loc[
    adata.obs["cell_area_um2_for_density"] <= 0,
    "cell_area_um2_for_density"
] = np.nan

Cell Size summary:
count    297668.000000
mean       1195.876910
std         557.189955
min         266.000000
25%         774.000000
50%        1101.000000
75%        1517.000000
max        3602.000000
Name: Cell Size, dtype: float64


In [139]:
# =========================
# 5b-3. Calculate CAR-T cells/mm2 by ROI and sample
# =========================
#group_col ='roi'
roi_col = "ROI index"

if roi_col not in adata.obs.columns:
    raise ValueError(f"没有找到 {roi_col}，请检查 obs columns。")

# 避免 sample_col == group_col 导致 reset_index 报错
roi_group_keys = list(dict.fromkeys([sample_col, group_col, roi_col]))
sample_group_keys = list(dict.fromkeys([sample_col, group_col]))

roi_density = (
    adata.obs
    .groupby(roi_group_keys, observed=False)
    .agg(
        analyzed_area_um2=("cell_area_um2_for_density", "sum"),
        total_segmented_cells=("CAR_T", "size"),
        CAR_T_cells=("CAR_T", "sum"),
        mCherry_pos_cells=("CAR_T_mCherry_pos", "sum"),
        GFP_pos_tumor_cells=("Tumor_GFP_pos", "sum"),
        CDH5_pos_segments=("Vascular_CDH5_pos", "sum")
    )
    .reset_index()
)

roi_density["analyzed_area_mm2"] = roi_density["analyzed_area_um2"] / 1e6

roi_density["CAR_T_cells_per_mm2"] = (
    roi_density["CAR_T_cells"] / roi_density["analyzed_area_mm2"]
)

roi_density["GFP_tumor_cells_per_mm2"] = (
    roi_density["GFP_pos_tumor_cells"] / roi_density["analyzed_area_mm2"]
)

roi_density["CAR_T_to_GFP_tumor_ratio"] = (
    roi_density["CAR_T_cells"] /
    roi_density["GFP_pos_tumor_cells"].replace(0, np.nan)
)

roi_density.to_csv(
    os.path.join(density_dir, "CAR_T_cells_per_mm2_by_ROI.csv"),
    index=False
)

sample_density_from_roi = (
    roi_density
    .groupby(sample_group_keys, observed=False)
    .agg(
        n_ROI=(roi_col, "nunique"),
        total_analyzed_area_mm2=("analyzed_area_mm2", "sum"),
        total_CAR_T_cells=("CAR_T_cells", "sum"),
        total_mCherry_pos_cells=("mCherry_pos_cells", "sum"),
        total_GFP_pos_tumor_cells=("GFP_pos_tumor_cells", "sum"),
        mean_CAR_T_cells_per_mm2=("CAR_T_cells_per_mm2", "mean"),
        median_CAR_T_cells_per_mm2=("CAR_T_cells_per_mm2", "median")
    )
    .reset_index()
)

sample_density_from_roi["pooled_CAR_T_cells_per_mm2"] = (
    sample_density_from_roi["total_CAR_T_cells"] /
    sample_density_from_roi["total_analyzed_area_mm2"]
)

sample_density_from_roi.to_csv(
    os.path.join(density_dir, "CAR_T_cells_per_mm2_sample_summary_from_ROI.csv"),
    index=False
)

print(roi_density.head())
print(sample_density_from_roi.head())

              sample ROI_group  ROI index  analyzed_area_um2  \
0     KA_CART_ROI2_1      CART          0         18910834.0   
1     KA_CART_ROI2_2      CART          0         41759383.0   
2  KA_CART_ROI2_3_c3      CART          0         14514882.0   
3  KA_CART_ROI2_3_c4      CART          0         15699686.0   
4     KA_CART_ROI2_4      CART          0         23807011.0   

   total_segmented_cells  CAR_T_cells  mCherry_pos_cells  GFP_pos_tumor_cells  \
0                  16118            0                  0                    0   
1                  39095           19                253                 6281   
2                  10557           21                348                  915   
3                  11228           29                370                 1009   
4                  18878            1                  3                    2   

   CDH5_pos_segments  analyzed_area_mm2  CAR_T_cells_per_mm2  \
0               3224          18.910834             0.000000   


In [144]:
# =========================
# 5c. Estimate analyzed tissue area
# =========================
#group_col = "roi"
group_col = 'ROI_group'
# 你需要根据实际 obs columns 修改
area_col = "Cell Size"
#roi_col = "ROI index"
roi_col='group'
sample_col="sample"


# 如果坐标是 pixel，而面积是 pixel，需要 pixel size
# 请按 MACSima 图像实际参数修改
pixel_size_um = 0.325  # example only; replace with real pixel size
pixel_area_um2 = pixel_size_um ** 2

if area_col is not None:
    print("Using cell area column:", area_col)

    if "px" in area_col.lower() or area_col.lower() in ["area", "cell_area", "area_px", "cell_area_px"]:
        adata.obs["cell_area_um2_for_density"] = pd.to_numeric(
            adata.obs[area_col], errors="coerce"
        ) * pixel_area_um2
    else:
        adata.obs["cell_area_um2_for_density"] = pd.to_numeric(
            adata.obs[area_col], errors="coerce"
        )

    area_by_sample = (
        adata.obs
        .groupby([sample_col, group_col], observed=False)
        .agg(
            analyzed_area_um2=("cell_area_um2_for_density", "sum"),
            total_segmented_cells=("CAR_T", "size")
        )
        .reset_index()
    )

else:
    print("Warning: 没有找到 cell area 列，将使用 spatial bounding box 估算 ROI 面积。")
    print("这比 tissue mask/cell area 弱，建议优先使用 tissue mask 或 segment area。")

    coords = adata.obsm["spatial"]
    adata.obs["_x_for_area"] = coords[:, 0]
    adata.obs["_y_for_area"] = coords[:, 1]

    area_records = []
    for (sample, group), df in adata.obs.groupby([sample_col, group_col], observed=False):
        width_px = df["_x_for_area"].max() - df["_x_for_area"].min()
        height_px = df["_y_for_area"].max() - df["_y_for_area"].min()
        roi_area_um2 = width_px * height_px * pixel_area_um2

        area_records.append({
            sample_col: sample,
            group_col: group,
            "analyzed_area_um2": roi_area_um2,
            "total_segmented_cells": len(df)
        })

    area_by_sample = pd.DataFrame(area_records)

area_by_sample["analyzed_area_mm2"] = area_by_sample["analyzed_area_um2"] / 1e6

area_by_sample.to_csv(
    os.path.join(density_dir, "analyzed_area_by_sample.csv"),
    index=False
)

Using cell area column: Cell Size


In [143]:
adata.obs[['ROI_group','group','ROI_index']]

KeyError: "['ROI_index'] not in index"

In [145]:
# =========================
# 5e. Vascular-associated versus extravascular CAR-T
# =========================
#sample_col='ROI'
#group_col='condition'
pixel_size_um = 0.325
pixel_area_um2 = pixel_size_um ** 2  # 0.105625 µm²/pixel
perivascular_threshold_um = 20 # 根据你的图像分辨率和生物学定义修改

if cdh5_marker is not None:
    coords_all = adata.obsm["spatial"]

    # 判断坐标是否需要 pixel-to-um 转换
    coords_um_all = coords_all * pixel_size_um

    vascular_records = []
    car_t_distance_records = []

    for (sample, group), df in adata.obs.groupby([sample_col, group_col], observed=False):
        idx = df.index
        pos = adata.obs.index.get_indexer(idx)
        coords_um = coords_um_all[pos, :]

        df_local = df.copy()
        df_local["_x_um"] = coords_um[:, 0]
        df_local["_y_um"] = coords_um[:, 1]

        car_t_df = df_local[df_local["CAR_T"]].copy()
        vessel_df = df_local[df_local["Vascular_CDH5_pos"]].copy()

        if len(car_t_df) == 0:
            vascular_records.append({
                sample_col: sample,
                group_col: group,
                "total_CAR_T": 0,
                "vascular_associated_CAR_T": 0,
                "extravascular_CAR_T": 0,
                "median_distance_to_CDH5_um": np.nan
            })
            continue

        if len(vessel_df) == 0:
            car_t_df["distance_to_nearest_CDH5_um"] = np.nan
            car_t_df["vascular_associated"] = False
            car_t_df["extravascular"] = np.nan
        else:
            vessel_coords = vessel_df[["_x_um", "_y_um"]].values
            car_t_coords = car_t_df[["_x_um", "_y_um"]].values

            tree = cKDTree(vessel_coords)
            dist_um, nearest_idx = tree.query(car_t_coords, k=1)

            car_t_df["distance_to_nearest_CDH5_um"] = dist_um
            car_t_df["vascular_associated"] = (
                car_t_df["distance_to_nearest_CDH5_um"] <= perivascular_threshold_um
            )
            car_t_df["extravascular"] = (
                car_t_df["distance_to_nearest_CDH5_um"] > perivascular_threshold_um
            )

        car_t_distance_records.append(car_t_df)

        vascular_records.append({
            sample_col: sample,
            group_col: group,
            "total_CAR_T": len(car_t_df),
            "vascular_associated_CAR_T": int(car_t_df["vascular_associated"].sum()),
            "extravascular_CAR_T": int(car_t_df["extravascular"].sum(skipna=True)),
            "median_distance_to_CDH5_um": car_t_df["distance_to_nearest_CDH5_um"].median()
        })

    vascular_summary = pd.DataFrame(vascular_records)

    vascular_summary = vascular_summary.merge(
        area_by_sample[[sample_col, group_col, "analyzed_area_mm2"]],
        on=[sample_col, group_col],
        how="left"
    )

    vascular_summary["vascular_associated_CAR_T_per_mm2"] = (
        vascular_summary["vascular_associated_CAR_T"] /
        vascular_summary["analyzed_area_mm2"]
    )

    vascular_summary["extravascular_CAR_T_per_mm2"] = (
        vascular_summary["extravascular_CAR_T"] /
        vascular_summary["analyzed_area_mm2"]
    )

    vascular_summary["extravascular_fraction"] = (
        vascular_summary["extravascular_CAR_T"] /
        vascular_summary["total_CAR_T"].replace(0, np.nan)
    )

    vascular_summary["vascular_associated_fraction"] = (
        vascular_summary["vascular_associated_CAR_T"] /
        vascular_summary["total_CAR_T"].replace(0, np.nan)
    )

    vascular_summary.to_csv(
        os.path.join(density_dir, "CAR_T_vascular_extravascular_summary_by_sample.csv"),
        index=False
    )

    if len(car_t_distance_records) > 0:
        car_t_distance_df = pd.concat(car_t_distance_records, axis=0)
        car_t_distance_df.to_csv(
            os.path.join(density_dir, "CAR_T_distance_to_CDH5_cell_level.csv"),
            index=True
        )

    print(vascular_summary)

                      sample       ROI_group  total_CAR_T  \
0             KA_CART_ROI2_1            CART            0   
1             KA_CART_ROI2_2            CART           19   
2          KA_CART_ROI2_3_c3            CART           21   
3          KA_CART_ROI2_3_c4            CART           29   
4             KA_CART_ROI2_4            CART            1   
5             KA_CART_ROI2_5            CART           71   
6             KA_CART_ROI2_6            CART            7   
7             KA_CART_ROI2_7            CART           18   
8   KA_inhibitor_0307_ROI2_2  inhibitor_0307            0   
9   KA_inhibitor_0307_ROI2_3  inhibitor_0307            1   
10  KA_inhibitor_0307_ROI2_4  inhibitor_0307            0   
11  KA_inhibitor_0307_ROI2_5  inhibitor_0307            2   

    vascular_associated_CAR_T  extravascular_CAR_T  \
0                           0                    0   
1                          16                    3   
2                           0               

In [146]:
# =========================
# 5f. Plot density results
# =========================

import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

def plot_group_scatter(df, y, ylabel, filename):
    groups = sorted(df[group_col].dropna().unique())
    x_map = {g: i for i, g in enumerate(groups)}

    fig, ax = plt.subplots(figsize=(3.2, 4))

    for g in groups:
        vals = df.loc[df[group_col] == g, y].dropna().values
        x = np.repeat(x_map[g], len(vals))
        jitter = np.random.normal(0, 0.04, size=len(vals))
        ax.scatter(x + jitter, vals, s=35)

        if len(vals) > 0:
            mean = np.mean(vals)
            sem = np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0
            ax.errorbar(
                x_map[g], mean, yerr=sem,
                fmt="_", capsize=4, linewidth=1.5
            )

    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(groups, rotation=45, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # 两组时添加 Mann-Whitney p value
    if len(groups) == 2:
        v1 = df.loc[df[group_col] == groups[0], y].dropna().values
        v2 = df.loc[df[group_col] == groups[1], y].dropna().values
        if len(v1) > 0 and len(v2) > 0:
            stat, p = mannwhitneyu(v1, v2, alternative="two-sided")
            ax.text(
                0.5, 0.95,
                f"p = {p:.4g}",
                transform=ax.transAxes,
                ha="center",
                va="top"
            )

    fig.tight_layout()
    fig.savefig(os.path.join(density_dir, filename), dpi=300, bbox_inches="tight")
    plt.close(fig)

'''
plot_group_scatter(
    #density_summary,
    sample_density_from_roi,
    y="CAR_T_cells_per_mm2",
    ylabel="CAR-T cells / mm² tumor tissue",
    filename="CAR_T_cells_per_mm2.png"
)

plot_group_scatter(
    #density_summary,
    sample_density_from_roi,
    y="CAR_T_to_tumor_ratio",
    ylabel="CAR-T / GFP+ tumor cell ratio",
    filename="CAR_T_to_tumor_ratio.png"
)
'''

if cdh5_marker is not None:
    plot_group_scatter(
        vascular_summary,
        y="extravascular_CAR_T_per_mm2",
        ylabel="Extravascular CAR-T cells / mm²",
        filename="extravascular_CAR_T_cells_per_mm2.png"
    )

    plot_group_scatter(
        vascular_summary,
        y="extravascular_fraction",
        ylabel="Extravascular CAR-T fraction",
        filename="extravascular_CAR_T_fraction.pdf"
    )

    plot_group_scatter(
        vascular_summary,
        y="median_distance_to_CDH5_um",
        ylabel="Median distance to CDH5+ vessel, μm",
        filename="CAR_T_median_distance_to_CDH5_um.pdf"
    )

In [147]:
# =========================
# 6. 基于 z-score marker 重新做 Leiden
# =========================

sc.pp.neighbors(
    adata,
    n_neighbors=n_neighbors_leiden,
    use_rep="X_marker_zscore",
    key_added="marker_zscore_neighbors"
)

sc.tl.leiden(
    adata,
    resolution=leiden_resolution,
    key_added="leiden_marker_zscore",
    neighbors_key="marker_zscore_neighbors"
)

print(adata.obs["leiden_marker_zscore"].value_counts())


# =========================
# 7. 输出每个 sample 的空间分布图
#    类似 ggplot2: coord_fixed() + scale_y_reverse()
# =========================

plot_cols = ["leiden_marker_zscore"] + zscore_cols

for sample in adata.obs[sample_col].unique():
    adata_sub = adata[adata.obs[sample_col] == sample].copy()

    # 7.1 Leiden 空间图
    fig = sc.pl.embedding(
        adata_sub,
        basis="spatial",
        color="leiden_marker_zscore",
        size=20,
        title=f"{sample} - Leiden based on marker z-score",
        show=False,
        return_fig=True
    )

    ax = fig.axes[0]
    ax.set_aspect("equal", adjustable="box")
    ax.invert_yaxis()

    fname = f"{safe_name(sample)}_leiden_marker_zscore.png"
    fig.savefig(
        os.path.join(plot_dir, fname),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

    # 7.2 每个 marker 的 z-score 空间图
    for marker, zcol, mshort in zip(marker_cols, zscore_cols, marker_short_names):
        fig = sc.pl.embedding(
            adata_sub,
            basis="spatial",
            color=zcol,
            size=20,
            title=f"{sample} - {mshort} z-score",
            show=False,
            return_fig=True,
            cmap="viridis"
        )

        ax = fig.axes[0]
        ax.set_aspect("equal", adjustable="box")
        ax.invert_yaxis()

        fname = f"{safe_name(sample)}_{safe_name(mshort)}_zscore.png"
        fig.savefig(
            os.path.join(plot_dir, fname),
            dpi=300,
            bbox_inches="tight"
        )
        plt.close(fig)

print(f"Spatial plots saved to: {plot_dir}")


# =========================
# 8. Leiden cluster 的 marker profile
# =========================

cluster_profile = (
    adata.obs
    .groupby("leiden_marker_zscore", observed=False)[zscore_cols]
    .mean()
)

cluster_profile.columns = marker_short_names

cluster_profile.to_csv(
    os.path.join(result_dir, "leiden_marker_zscore_cluster_profile.csv")
)

# cluster marker heatmap
fig, ax = plt.subplots(figsize=(max(8, len(marker_cols) * 0.4), 5))
im = ax.imshow(cluster_profile.values, aspect="auto")
ax.set_xticks(np.arange(len(cluster_profile.columns)))
ax.set_xticklabels(cluster_profile.columns, rotation=90)
ax.set_yticks(np.arange(cluster_profile.shape[0]))
ax.set_yticklabels(cluster_profile.index)
ax.set_xlabel("Marker")
ax.set_ylabel("Leiden cluster")
ax.set_title("Mean marker z-score per Leiden cluster")
fig.colorbar(im, ax=ax, label="Mean z-score")
fig.tight_layout()
fig.savefig(
    os.path.join(heatmap_dir, "leiden_marker_zscore_cluster_profile_heatmap.pdf"),
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)


# =========================
# 9. 空间共定位分析
# =========================
# 定义：
# 对每个 sample 内的细胞构建空间 kNN。
# 对 marker A 和 marker B：
#   score(A,B) = 0.5 * [
#       z_A(cell) * mean_neighbor_z_B(cell)
#       +
#       z_B(cell) * mean_neighbor_z_A(cell)
#   ]
#
# score 越高，表示 A 高表达细胞附近更倾向于出现 B 高表达，反之亦然。
# 这个是局部空间共定位 score，不是普通全局表达相关性。

pair_records = []
cell_level_records = []

coords_all = adata.obsm["spatial"]
X_all = adata.obsm["X_marker_zscore"]

for sample in adata.obs[sample_col].unique():
    idx = np.where(adata.obs[sample_col].values == sample)[0]

    if len(idx) <= spatial_k:
        print(f"Skip {sample}: cell number {len(idx)} <= spatial_k {spatial_k}")
        continue

    coords = coords_all[idx, :]
    X = X_all[idx, :]

    group_value = adata.obs.iloc[idx][group_col].iloc[0]

    nn = NearestNeighbors(
        n_neighbors=spatial_k + 1,
        metric="euclidean"
    )
    nn.fit(coords)
    distances, neighbors = nn.kneighbors(coords)

    # 去掉自己
    neighbors = neighbors[:, 1:]
    distances = distances[:, 1:]

    # 每个细胞邻域内各 marker 的平均 z-score
    neighbor_mean = np.zeros_like(X)
    for i in range(X.shape[0]):
        neighbor_mean[i, :] = X[neighbors[i], :].mean(axis=0)

    for a in range(len(marker_cols)):
        for b in range(a + 1, len(marker_cols)):
            score_vec = 0.5 * (
                X[:, a] * neighbor_mean[:, b]
                +
                X[:, b] * neighbor_mean[:, a]
            )

            marker_a = marker_short_names[a]
            marker_b = marker_short_names[b]
            pair_name = f"{marker_a}__{marker_b}"

            pair_records.append({
                "sample": sample,
                "group": group_value,
                "marker_a": marker_a,
                "marker_b": marker_b,
                "pair": pair_name,
                "n_cells": len(score_vec),
                "mean_coloc_score": np.nanmean(score_vec),
                "median_coloc_score": np.nanmedian(score_vec),
                "std_coloc_score": np.nanstd(score_vec),
                "q25_coloc_score": np.nanquantile(score_vec, 0.25),
                "q75_coloc_score": np.nanquantile(score_vec, 0.75),
            })

            # 保存 cell-level 分布，用于后续画分布图或更细分析
            # 如果细胞很多、marker 很多，这个文件会比较大。
            for val in score_vec:
                cell_level_records.append({
                    "sample": sample,
                    "group": group_value,
                    "marker_a": marker_a,
                    "marker_b": marker_b,
                    "pair": pair_name,
                    "coloc_score": val
                })

coloc_summary = pd.DataFrame(pair_records)
coloc_cell_level = pd.DataFrame(cell_level_records)

coloc_summary.to_csv(
    os.path.join(result_dir, "spatial_colocalization_summary_by_sample.csv"),
    index=False
)

coloc_cell_level.to_csv(
    os.path.join(result_dir, "spatial_colocalization_cell_level_scores.csv"),
    index=False
)

print("Colocalization summary:")
print(coloc_summary.head())


# =========================
# 10. 组间共定位差异分析
# =========================
# 优先使用 sample-level summary 做组间比较。
# 如果 group_col == sample，相当于比较不同 sample。
# 如果你有 group / condition，则是比较不同 biological groups。

diff_records = []

for pair, df_pair in coloc_summary.groupby("pair"):
    groups = sorted(df_pair["group"].dropna().unique())

    if len(groups) < 2:
        continue

    values_by_group = [
        df_pair.loc[df_pair["group"] == g, "mean_coloc_score"].dropna().values
        for g in groups
    ]

    # 至少每组有一个 sample
    if any(len(v) == 0 for v in values_by_group):
        continue

    if len(groups) == 2:
        g1, g2 = groups
        v1 = values_by_group[0]
        v2 = values_by_group[1]

        if len(v1) >= 1 and len(v2) >= 1:
            stat, pval = mannwhitneyu(v1, v2, alternative="two-sided")

            diff_records.append({
                "pair": pair,
                "marker_a": df_pair["marker_a"].iloc[0],
                "marker_b": df_pair["marker_b"].iloc[0],
                "test": "Mann-Whitney U",
                "group_1": g1,
                "group_2": g2,
                "mean_group_1": np.mean(v1),
                "mean_group_2": np.mean(v2),
                "difference_group2_minus_group1": np.mean(v2) - np.mean(v1),
                "statistic": stat,
                "pvalue": pval,
                "n_group_1": len(v1),
                "n_group_2": len(v2),
            })

    else:
        # 多组时用 Kruskal-Wallis
        if all(len(v) >= 1 for v in values_by_group):
            stat, pval = kruskal(*values_by_group)

            group_means = {
                f"mean_{g}": np.mean(v)
                for g, v in zip(groups, values_by_group)
            }

            diff_records.append({
                "pair": pair,
                "marker_a": df_pair["marker_a"].iloc[0],
                "marker_b": df_pair["marker_b"].iloc[0],
                "test": "Kruskal-Wallis",
                "groups": ";".join(groups),
                "statistic": stat,
                "pvalue": pval,
                **group_means
            })

coloc_diff = pd.DataFrame(diff_records)

if len(coloc_diff) > 0:
    coloc_diff["fdr_bh"] = multipletests(
        coloc_diff["pvalue"].values,
        method="fdr_bh"
    )[1]

    coloc_diff = coloc_diff.sort_values("fdr_bh")

coloc_diff.to_csv(
    os.path.join(result_dir, "spatial_colocalization_group_difference.csv"),
    index=False
)

print("Top differential colocalization pairs:")
print(coloc_diff.head(20))


# =========================
# 11. 每个 sample 的 marker 相关性分析
# =========================
# 这里是普通表达相关性，不考虑空间邻居。
# 用于回答：同一个细胞层面，不同 marker 的表达是否相关。

corr_records = []

for sample in adata.obs[sample_col].unique():
    idx = np.where(adata.obs[sample_col].values == sample)[0]
    X = adata.obsm["X_marker_zscore"][idx, :]
    group_value = adata.obs.iloc[idx][group_col].iloc[0]

    for a in range(len(marker_cols)):
        for b in range(a + 1, len(marker_cols)):
            xa = X[:, a]
            xb = X[:, b]

            # 去掉 nan
            valid = np.isfinite(xa) & np.isfinite(xb)

            if valid.sum() < 5:
                continue

            rho, p_spearman = spearmanr(xa[valid], xb[valid])
            r, p_pearson = pearsonr(xa[valid], xb[valid])

            corr_records.append({
                "sample": sample,
                "group": group_value,
                "marker_a": marker_short_names[a],
                "marker_b": marker_short_names[b],
                "pair": f"{marker_short_names[a]}__{marker_short_names[b]}",
                "n_cells": valid.sum(),
                "spearman_rho": rho,
                "spearman_pvalue": p_spearman,
                "pearson_r": r,
                "pearson_pvalue": p_pearson
            })

corr_df = pd.DataFrame(corr_records)

if len(corr_df) > 0:
    corr_df["spearman_fdr_bh"] = multipletests(
        corr_df["spearman_pvalue"].fillna(1).values,
        method="fdr_bh"
    )[1]

    corr_df["pearson_fdr_bh"] = multipletests(
        corr_df["pearson_pvalue"].fillna(1).values,
        method="fdr_bh"
    )[1]

corr_df.to_csv(
    os.path.join(result_dir, "marker_correlation_by_sample.csv"),
    index=False
)

print("Marker correlation results:")
print(corr_df.head())


# =========================
# 12. 输出每个 sample 的 marker correlation heatmap
# =========================

for sample in adata.obs[sample_col].unique():
    idx = np.where(adata.obs[sample_col].values == sample)[0]
    X = adata.obsm["X_marker_zscore"][idx, :]

    if X.shape[0] < 5:
        continue

    corr_mat = pd.DataFrame(
        X,
        columns=marker_short_names
    ).corr(method="spearman")

    corr_mat.to_csv(
        os.path.join(
            result_dir,
            f"{safe_name(sample)}_spearman_marker_correlation_matrix.csv"
        )
    )

    fig, ax = plt.subplots(figsize=(max(8, len(marker_cols) * 0.45), max(6, len(marker_cols) * 0.45)))
    im = ax.imshow(corr_mat.values, vmin=-1, vmax=1)
    ax.set_xticks(np.arange(len(marker_short_names)))
    ax.set_xticklabels(marker_short_names, rotation=90)
    ax.set_yticks(np.arange(len(marker_short_names)))
    ax.set_yticklabels(marker_short_names)
    ax.set_title(f"{sample} - Spearman marker correlation")
    fig.colorbar(im, ax=ax, label="Spearman rho")
    fig.tight_layout()

    fig.savefig(
        os.path.join(
            heatmap_dir,
            f"{safe_name(sample)}_spearman_marker_correlation_heatmap.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)


# =========================
# 13. 输出每个 group 的平均共定位矩阵 heatmap
# =========================

for group_value, df_group in coloc_summary.groupby("group"):
    mat = pd.DataFrame(
        np.nan,
        index=marker_short_names,
        columns=marker_short_names
    )

    for _, row in df_group.groupby(["marker_a", "marker_b"])["mean_coloc_score"].mean().reset_index().iterrows():
        a = row["marker_a"]
        b = row["marker_b"]
        val = row["mean_coloc_score"]
        mat.loc[a, b] = val
        mat.loc[b, a] = val

    np.fill_diagonal(mat.values, 0)

    mat.to_csv(
        os.path.join(
            result_dir,
            f"{safe_name(group_value)}_mean_spatial_colocalization_matrix.csv"
        )
    )

    fig, ax = plt.subplots(figsize=(max(8, len(marker_cols) * 0.45), max(6, len(marker_cols) * 0.45)))
    im = ax.imshow(mat.values)
    ax.set_xticks(np.arange(len(marker_short_names)))
    ax.set_xticklabels(marker_short_names, rotation=90)
    ax.set_yticks(np.arange(len(marker_short_names)))
    ax.set_yticklabels(marker_short_names)
    ax.set_title(f"{group_value} - Mean spatial colocalization score")
    fig.colorbar(im, ax=ax, label="Mean colocalization score")
    fig.tight_layout()

    fig.savefig(
        os.path.join(
            heatmap_dir,
            f"{safe_name(group_value)}_mean_spatial_colocalization_heatmap.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)


# =========================
# 14. 保存重新分析后的 AnnData
# =========================

adata.write_h5ad(
    os.path.join(outdir, "all_ROI_spatial_reanalyzed_marker_density_zscore_leiden.h5ad")
)

print("Done.")
print(f"All results saved to: {outdir}")

leiden_marker_zscore
0     69481
1     68194
2     49253
3     48829
4     30944
5     13277
6      4038
7      3876
8      2845
9      1354
10     1295
11     1094
12      974
13      861
14      611
15      508
16      107
17       64
18       43
19       20
Name: count, dtype: int64
Spatial plots saved to: spatial_reanalysis_density/spatial_plots
Colocalization summary:
           sample group marker_a             marker_b  \
0  KA_CART_ROI2_1  CART     CCR1           CD4 REA623   
1  KA_CART_ROI2_1  CART     CCR1         CD8 BW135/80   
2  KA_CART_ROI2_1  CART     CCR1  CD11b M1/70.15.11.5   
3  KA_CART_ROI2_1  CART     CCR1             CD25 4E3   
4  KA_CART_ROI2_1  CART     CCR1           CD44 DB105   

                        pair  n_cells  mean_coloc_score  median_coloc_score  \
0           CCR1__CD4 REA623    16118          0.097043            0.035691   
1         CCR1__CD8 BW135/80    16118          0.111544            0.040763   
2  CCR1__CD11b M1/70.15.11.5    16118       